# Preprocesamiento: Dataset MINSA
Dataset: Vigilancia Epidemiológica del dengue
Enlace Original [Plataforma Nacional de Datos Abiertos](https://www.datosabiertos.gob.pe/dataset/vigilancia-epidemiol%C3%B3gica-de-dengue)

El Dataset muestra todos los casos de dengue confirmados, probables y sospechosos en el territorio peruano en el periodo 2000-2025.
El flujo describe todo el proceso de Extracción, Transformación y Carga para filtrar todos los datos de la región Piuram pues esta es la región perteneciente al Norte con mayor correlación entre el calentamiento superficial del mar, lluvias extremas y picos de dengue.

Históricamente Piura presenta los brotes de dengue más altos del país (especialmente entre 2017 y 2023).

Finalmente, en Piura existe la estación meteorológica de Miraflores en la Universidad Nacional de Piura, por lo que será accesible conectar los registros de este dataset con los registros de los datos de la estación.

## Importación de la librería pandas y carga del dataset original

In [1]:
from pathlib import Path
import pandas as pd

# 1. CARGA DEL DATASET MASIVO
ruta_pnda = next(
    (ruta / 'Datasets_PNDA' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'Datasets_PNDA').is_dir()),
    None
)
if ruta_pnda is None:
    raise FileNotFoundError("No se encontró la carpeta 'Datasets_PNDA' del proyecto.")

ruta_csv = ruta_pnda / 'datos_abiertos_vigilancia_dengue_2000_2025_0.csv'

# Usamos low_memory=False por si el archivo supera los cientos de miles de filas
df_minsa = pd.read_csv(ruta_csv, sep=',', encoding='utf-8', low_memory=False)

print(f"Total de registros crudos a nivel nacional: {len(df_minsa)}")
print(f"Dimensiones del dataset crudo: {df_minsa.shape}")

Total de registros crudos a nivel nacional: 1065274
Dimensiones del dataset crudo: (1065274, 15)


### Análisis Inicial
El dataset original presenta un total de 1065274 registros distribuidos en 15 columnas. La descripción de las columnas se presenta a continuación.

In [8]:
df_minsa.info()

<class 'pandas.DataFrame'>
RangeIndex: 1065274 entries, 0 to 1065273
Data columns (total 15 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   departamento  1065274 non-null  str    
 1   provincia     1065274 non-null  str    
 2   distrito      1065274 non-null  str    
 3   localidad     941639 non-null   str    
 4   enfermedad    1065274 non-null  str    
 5   ano           1065274 non-null  int64  
 6   semana        1065274 non-null  int64  
 7   diagnostic    1065274 non-null  str    
 8   tipo_dx       1065274 non-null  str    
 9   diresa        1065251 non-null  float64
 10  ubigeo        1065274 non-null  int64  
 11  localcod      915622 non-null   str    
 12  edad          1065274 non-null  int64  
 13  tipo_edad     1065274 non-null  str    
 14  sexo          1065274 non-null  str    
dtypes: float64(1), int64(4), str(10)
memory usage: 121.9 MB


La serie de columnas muestra todos las variables demográficas de cada uno de los pacientes registrados, las variables útiles para este proyecto serán departamento, ano, semana y tipo_dx

departamento: Contiene los valores respectivos de cada uno de los departamentos del Perú.

ano: Registra cada paciente notificado desde el periodo 2000 a 2025

semana: Registra cada paciente en cada una de las 53 semanas del año

tipo_dx: Almacena el diagnóstico final de cada paciente: C= Confirmado, P = Probable, S = Sospechoso


Para fines de este proyecto, se utilizará el departamento de Piura, correspondiente al periodo completo de 25 años, contemplando las 53 semanas de cada año y filtrando únicamente los casos confirmados de dengue.

In [3]:
# 2. FILTRADO ESTRATÉGICO (Región y Calidad de Diagnóstico)
# Filtramos por el departamento de PIURA y solo casos Confirmados ('C')
df_piura = df_minsa[
    (df_minsa['departamento'].str.upper() == 'PIURA') & 
    (df_minsa['tipo_dx'] == 'C')
].copy()

print(f"Casos confirmados aislados en Piura: {len(df_piura)}")
print(f"Dimensiones del dataset filtrado: {df_piura.shape}")
display(df_piura.head(10))

Casos confirmados aislados en Piura: 217048
Dimensiones del dataset filtrado: (217048, 15)


,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,tipo_dx,diresa,ubigeo,localcod,edad,tipo_edad,sexo
2581,PIURA,SULLANA,SULLANA,SULLANA,DENGUE SIN SIGNOS DE ALARMA,2000,25,A97.0,C,31.0,200601,NaN,13,A,F
2582,PIURA,SULLANA,SULLANA,NaN,DENGUE SIN SIGNOS DE ALARMA,2000,23,A97.0,C,31.0,200601,NaN,15,A,F
2583,PIURA,SULLANA,SULLANA,EL OBRERO,DENGUE SIN SIGNOS DE ALARMA,2000,24,A97.0,C,31.0,200601,NaN,50,A,M
2584,PIURA,SULLANA,SULLANA,BUENOS AIRES,DENGUE SIN SIGNOS DE ALARMA,2000,24,A97.0,C,31.0,200601,NaN,62,A,F
2585,PIURA,SULLANA,SULLANA,SANTA ROSA,DENGUE SIN SIGNOS DE ALARMA,2000,23,A97.0,C,31.0,200601,NaN,25,A,M
2586,PIURA,SULLANA,SULLANA,NUEVE DE OCTUBRE,DENGUE SIN SIGNOS DE ALARMA,2000,23,A97.0,C,31.0,200601,NaN,25,A,F
2587,PIURA,SULLANA,SULLANA,SANCHEZ CERRO,DENGUE SIN SIGNOS DE ALARMA,2000,22,A97.0,C,31.0,200601,NaN,33,A,M
2588,PIURA,SULLANA,SULLANA,EL OBRERO,DENGUE SIN SIGNOS DE ALARMA,2000,24,A97.0,C,31.0,200601,NaN,30,A,M
2589,PIURA,SULLANA,SULLANA,NUEVE DE OCTUBRE,DENGUE SIN SIGNOS DE ALARMA,2000,25,A97.0,C,31.0,200601,NaN,27,A,M
2590,PIURA,SULLANA,SULLANA,NUEVE DE OCTUBRE,DENGUE SIN SIGNOS DE ALARMA,2000,25,A97.0,C,31.0,200601,NaN,52,A,M


### Segundo análisis exploratorio

Una vez filtrado todo el periodo para la región Piura, el dataset resultante contiene 217048 registros con las mismas 15 columnas.

Para una visualización más sencilla de los casos de dengue, a continuación se muestran todos los casos en las primeras 10 semanas del año 2025

In [4]:
df_piura[df_piura['ano'] == 2025]['semana'].value_counts().sort_index().head(10)  # Contamos los casos por semana y ordenamos por semana

semana
1     33
2     48
3     45
4     58
5     40
6     50
7     53
8     40
9     46
10    45
Name: count, dtype: int64

## Transformación a series de tiempo semanales
Como cada fila es un paciente, contar las filas por año y semana nos da el total de casos. Luego, se ordenan todos los registros en forma cronológica, esto será de utilidad para el modelo.

In [5]:
# 3. AGREGACIÓN 
serie_dengue = df_piura.groupby(['ano', 'semana']).size().reset_index(name='Casos_Confirmados')

# Ordenamos cronológicamente de forma estricta 
serie_dengue = serie_dengue.sort_values(by=['ano', 'semana']).reset_index(drop=True)

print(serie_dengue)


       ano  semana  Casos_Confirmados
0     2000      21                  6
1     2000      22                 29
2     2000      23                 57
3     2000      24                 89
4     2000      25                 49
...    ...     ...                ...
1020  2025      49                  3
1021  2025      50                  2
1022  2025      51                  3
1023  2025      52                  4
1024  2025      53                  3

[1025 rows x 3 columns]


## Imputación de datos nulos/ vacíos
En el contexto del proyecto, las semanas que no presenten casos de dengue se rellenarán con valores 0 y así no perder el registro histórico completo

In [6]:
# 4. RELLENO DE SEMANAS VACÍAS (Continuidad de la señal)
# Si en una semana de invierno hubo 0 casos, no aparecerá en el CSV. 
# Para una serie de tiempo, un hueco destruye la ventana deslizante; se rellena con 0.
anos_unicos = range(serie_dengue['ano'].min(), serie_dengue['ano'].max() + 1)
semanas_unicas = range(1, 53) # 52 semanas estándar por año

# Creamos un calendario maestro completo y cruzamos con nuestros datos reales (how='outer' conserva la semana 53)
indice_completo = pd.MultiIndex.from_product([anos_unicos, semanas_unicas], names=['ano', 'semana']).to_frame().reset_index(drop=True)
df_target_final = pd.merge(indice_completo, serie_dengue, on=['ano', 'semana'], how='outer').sort_values(by=['ano', 'semana']).reset_index(drop=True)

# Los NaN significan que no hubo reportes esa semana -> Se convierten en 0 casos
df_target_final['Casos_Confirmados'] = df_target_final['Casos_Confirmados'].fillna(0).astype(int)

print("\n--- VISTA PREVIA DE TU VARIABLE OBJETIVO (Y) LISTA ---")
display(df_target_final.tail(10))


--- VISTA PREVIA DE TU VARIABLE OBJETIVO (Y) LISTA ---


,ano,semana,Casos_Confirmados
1346,2025,44,6
1347,2025,45,10
1348,2025,46,11
1349,2025,47,8
1350,2025,48,4
1351,2025,49,3
1352,2025,50,2
1353,2025,51,3
1354,2025,52,4
1355,2025,53,3


## Exportación del dataset final
Finalmente, el siguiente código exporta el dataset resultante a la carpeta Datasets para su posterior uso.

In [9]:
# Guardar archivo CSV final en la carpeta 'Datasets' del proyecto
from pathlib import Path
ruta_datasets = next(
    (ruta / 'Datasets' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'Datasets').is_dir()),
    None
)
if ruta_datasets is None:
    raise FileNotFoundError("No se encontró la carpeta 'Datasets' del proyecto.")

ruta_csv = ruta_datasets / 'casos_dengue_piura_2000_2025_semanal.csv'
df_target_final.to_csv(ruta_csv, index=False)
print(f"Dataset guardado en: {ruta_csv}")

Dataset guardado en: c:\Users\fabri\OneDrive\Desktop\PROYECTO-IA\Datasets\casos_dengue_piura_2000_2025_semanal.csv
